# Check out Parameter Calibration

In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd

import fates_calibration_library.utils as utils
import fates_calibration_library.clm_functions as clm
import fates_calibration_library.emulator_functions as em
from fates_calibration_library.TFClass import TFEmulator

import matplotlib.pyplot as plt
import importlib

## Set Up
Load files, set up ensemble information

In [ ]:
def plot_parameter_hists(df, default_pft):
    
    plt.figure(figsize=[18, 16])
    pars = df.columns
    for i, par in enumerate(pars):
        p = default_pft[par].values
        ax = plt.subplot(7, 5, i + 1)
        ax.hist(df[par])
        ax.axvline(x=p, color='r', linestyle='-')
        ax.set_xlabel(par)
        ax.set_xlim(0, 1)
    plt.tight_layout()

def get_corr(sample_df):
    corr = sample_df.corr()
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask)] = True
    corr[mask] = np.nan
    return corr

In [ ]:
# directories
mesh_dir = '/glade/work/afoster/FATES_calibration/mesh_files'
emulator_dir = '/glade/work/afoster/FATES_calibration/emulators'
fig_dir = '/glade/work/afoster/FATES_calibration/figures'
param_dir = '/glade/work/afoster/FATES_calibration/parameter_files'
out_dir = '/glade/work/afoster/FATES_calibration/parameter_outputs'

# default parameter file
default_param = xr.open_dataset(os.path.join(param_dir, 
                                             'fates_params_default_sci.1.85.1_api.40.0.0_crops.nc'))
all_pfts = [str(pft).replace("b'", "").replace("'", "").strip() for pft in default_param.fates_pftname.values]

# normalized values for parameters
default_norm = pd.read_csv(os.path.join(param_dir, 'normalized_parameters.csv'), index_col=[0])

# variables to calibrate
calibration_vars = ['GPP', 'EFLX_LH_TOT', 'FSH', 'EF']

# information about variables
obs_config_file = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/ilamb_conversion.yaml'
obs_config = utils.get_config_file(obs_config_file)

# PFT ids
pft_id_config = '/glade/work/afoster/FATES_calibration/fates_calibration_library/configs/fates_pft_ids.yaml'
pft_ids = utils.get_config_file(pft_id_config)

In [ ]:
# information about each ensemble
ens_dict = {'dompft':
            {'mesh_file': os.path.join(mesh_dir, 'dominant_grid_mesh.nc'),
             'land_mask_file': os.path.join(mesh_dir, 'dominant_grid.nc'),
             'lhc_key_file': os.path.join(param_dir, 'fates_lh', 'fates_lh_key.csv'),
             'pfts': [1, 2, 3, 12, 13, 14],
             'obs_df': os.path.join(mesh_dir, 'dominant_grid.csv'),
            }
           }

In [ ]:
# choose ensemble
ensemble = 'dompft'

In [ ]:
lhc_key = pd.read_csv(ens_dict[ensemble]['lhc_key_file'], index_col=[0])
lhc_key = lhc_key.drop(columns=['ensemble'])
param_names = lhc_key.columns
num_params = len(param_names)

In [ ]:
# load observations
obs = pd.read_csv(ens_dict[ensemble]['obs_df'], index_col=[0])

In [ ]:
pft = 1
pft_name = all_pfts[pft-1]
pft_id = pft_ids[pft_name]

In [ ]:
# get observations for this pft
obs_pft = obs[obs.pft == pft_name]

# get default values for this pft
default_pft = default_norm[default_norm.pft == pft]
default_pft = default_pft.drop(columns=['pft'])
X_default_all = default_pft.to_numpy().flatten()

In [ ]:
# stack targets, sds, and emulators for all variables
targets = []
sds = []
emulators = []
for variable in calibration_vars:
    
    # observations for this pft and variable
    obs_mean, obs_sd = em.get_obs_mean_and_sd(obs_pft, obs_config[variable]['var'])
    
    # convert to tf objects
    targets.append(obs_mean)
    sds.append(obs_sd)

    # load the emulator
    emulators.append(TFEmulator(emulator_dir, pft=pft_id, variable=variable))

In [ ]:
# get "final" parameters
param_files = [os.path.join(out_dir, f) for f in os.listdir(out_dir) if f.endswith('.csv')]
dat_list = []
for file in param_files:
    dat_list.append(pd.read_csv(file))
dat = pd.concat(dat_list)
dat = dat.drop(columns=['batch'])

In [ ]:
plot_parameter_hists(dat, default_pft)

In [ ]:
fixed_indices = np.where(~np.isin(param_names.values, dat.columns.values))[0]

In [ ]:
sample = np.asarray(dat)
result = np.array([
    em.get_full_array(sample[i], fixed_indices, X_default_all)
    for i in range(sample.shape[0])
])

In [ ]:
dat

In [ ]:
for i, variable in enumerate(calibration_vars):
    y_pred, y_var = emulators[i](result)
    implaus = em.implausibility_metric(y_pred.numpy().flatten(), targets[i],
                                       y_var.numpy().flatten(), sds[0]**2)
    dat[f"{variable}_implaus"] = implaus
    em.plot_implausibility_histogram(implaus, tol=1)
    em.plot_emulated_sample(y_pred.numpy().flatten(), targets[i], sds[i],
                            pft_id, calibration_vars[i], '')

In [ ]:
col_list = [f"{f}_implaus" for f in calibration_vars]

In [ ]:
dat_sub = em.subset_sample(dat, col_list, 1)

In [ ]:
dat_sub['sum_implaus'] = em.calculate_implaus_sum(dat_sub, col_list)

In [ ]:
dat_out = dat_sub.where(dat_sub['sum_implaus'] < 1)
dat_out = dat_out.dropna()
dat_out = dat_out.drop(columns=np.append('sum_implaus', col_list))

In [ ]:
corr = get_corr(dat_out)

In [ ]:
(corr.style.background_gradient(cmap='coolwarm', axis=None, vmin=-1, vmax=1).highlight_null(color='#f1f1f1').format(precision=2))

In [ ]:
plot_parameter_hists(dat_out, default_pft)
plt.savefig(os.path.join(fig_dir, 'param_hists.png'))

In [ ]:
var_choose = 'fates_leaf_vcmax25top'
var_median = dat_out[var_choose].median()
dat_out['var_diff'] = np.abs(dat_out[var_choose] - var_median)
choose_index = np.argmin(dat_out.var_diff)
dat_out = dat_out.drop(columns=['var_diff'])

In [ ]:
final_pars_scaled = dat_out.iloc[choose_index]

In [ ]:
final_pars_scaled

In [ ]:
plt.scatter(dat_out.fates_rad_leaf_clumping_index, dat_out.fates_leaf_vcmax25top)